# 🕵️ Data Detective Lab — Day 4
### Turning your Day 2 sensor data into a story for tomorrow's pitch

**Today's agenda (10:00 – 12:00):**
1. Common toolkit — the moves every data detective uses, no matter the case (10:10–10:35)
2. Break into your squad's section below and apply the toolkit to *your* data (10:40–11:25)
3. Practice your 2-minute "here's what our data told us" pitch (11:25–11:45)

Run each code cell with **Shift + Enter**. Don't worry about breaking anything — you can always re-run a cell.

**A note on the data:** every squad section below starts with a sample dataset shaped like real sensor output (Pulse Sensor Amped, DS18B20 temperature sensor, a water-quality field survey, and a generic PM2.5 monitor) so today's cells run right now. Swap in your own Day 2 CSV the moment it's ready — just change the filename in the load cell.


## Part 1 — Setup

Every investigation starts with the same three tools: `pandas` for tables, `numpy` for numbers, `matplotlib` for pictures.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
print("Ready to detect! 🔍")


## Part 2 — Load & Explore ANY Dataset

Before you can find a clue, you have to know what's in the box. These four moves work on **any** CSV file:

| Move | Code | What it tells you |
|---|---|---|
| Peek at the top | `df.head()` | What columns exist, what the values look like |
| Check the shape/types | `df.info()` | How many rows, missing values, data types |
| Summarize the numbers | `df.describe()` | Min, max, average, spread |
| Plot it | `df.plot(...)` | Patterns, trends, weird spikes |

We'll practice on a **warm-up dataset** first (not any squad's real case) — a week of hourly readings from a single mystery sensor.


In [ ]:
rng = np.random.default_rng(42)
timestamps = pd.date_range("2026-07-01", periods=168, freq="H")
values = 20 + 3*np.sin(np.arange(168)/12) + rng.normal(0, 1, 168)
values[100] += 8  # plant one obvious anomaly

warmup = pd.DataFrame({"timestamp": timestamps, "reading": values})
warmup.head()


In [ ]:
warmup.info()


In [ ]:
warmup.describe()


In [ ]:
warmup.plot(x="timestamp", y="reading", title="Mystery Sensor Readings Over One Week")
plt.ylabel("Reading")
plt.show()


## Part 3 — The Data Detective Toolkit

Two techniques you'll use in your squad section, no matter your track:

**1. Flagging anomalies** — compare each value to a rolling average or a known safe threshold. If it's far enough off, flag it.

**2. Comparing groups** — use `groupby()` to compare readings across sites, conditions, or before/after states.


In [ ]:
# Technique 1: flag anomalies using a rolling average + threshold
warmup["rolling_avg"] = warmup["reading"].rolling(window=6, center=True).mean()
warmup["anomaly"] = (warmup["reading"] - warmup["rolling_avg"]).abs() > 3

fig, ax = plt.subplots()
ax.plot(warmup["timestamp"], warmup["reading"], label="reading")
ax.plot(warmup["timestamp"], warmup["rolling_avg"], label="rolling avg")
ax.scatter(warmup.loc[warmup["anomaly"], "timestamp"],
           warmup.loc[warmup["anomaly"], "reading"],
           color="red", zorder=5, label="anomaly")
ax.legend()
ax.set_title("Spotting the Anomaly")
plt.show()

print(f"Found {warmup['anomaly'].sum()} anomaly point(s).")


In [ ]:
# Technique 2: compare groups (here: morning vs afternoon vs night)
warmup["hour"] = warmup["timestamp"].dt.hour
def part_of_day(h):
    if h < 8: return "night"
    if h < 16: return "morning"
    return "evening"
warmup["period"] = warmup["hour"].apply(part_of_day)

warmup.groupby("period")["reading"].mean().plot(kind="bar", title="Average Reading by Time of Day")
plt.ylabel("Average reading")
plt.show()


---
## 🎒 Part 4 — Squad Sections

Only work in **your squad's section** below. Each one follows the same pattern:
1. Load your data (a sample file loads today; point it at your real Day 2 CSV when it's ready)
2. Describe it
3. Plot it
4. Flag something that matters for your case
5. Write your data story for tomorrow's pitch


### 💧 Squad 1: Water Quality

**Mission:** Runoff and pollution threaten our streams, lakes, and rivers. Your field tests along Minnehaha Creek measured things like pH, hardness, lead, and nitrate at several sites — but not every group tested every parameter, so this data has real gaps, just like your actual Day 2 file will.


In [ ]:
df = pd.read_csv("data/water_quality.csv")
# Swap the path above for your real Day 2 export when it's ready — same column names.
df.head()


**Notice:** some columns have missing values (`NaN`) — that's because different groups tested different parameters. `df.info()` will show you exactly which columns are more complete than others.


In [ ]:
df.info()


In [ ]:
df.describe()


**TODO:** Compare a parameter (start with `pH`) across sites. Try:
```python
df.groupby("Site ID")["pH"].mean().plot(kind="bar", title="Average pH by Site")
```


In [ ]:
# TODO: your groupby + plot here



**TODO — mini GIS map:** you have real coordinates (`x` = longitude, `y` = latitude)! Make a scatter plot colored by a parameter to see if contamination clusters in one part of the creek:
```python
plt.scatter(df["x"], df["y"], c=df["Lead (ppb)"], cmap="Reds")
plt.colorbar(label="Lead (ppb)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("Lead Levels Along Minnehaha Creek")
```


In [ ]:
# TODO: your mini-GIS scatter map here



**TODO — flag a concern:** safe drinking water is roughly pH 6.5–8.5, and the EPA action level for lead is 15 ppb. Which rows fall outside safe ranges?
```python
df[(df["pH"] < 6.5) | (df["pH"] > 8.5)]
df[df["Lead (ppb)"] > 15]
```


In [ ]:
# TODO: your flagged rows here



**Data story — answer these for tomorrow's pitch:**
- Which site(s) had the most concerning readings, and for which parameter?
- Does contamination look clustered in one part of the creek, or spread out?
- What would you recommend to address it?

*(Double-click this cell and type your answers right here.)*


### 🌬️ Squad 2: Air Quality

**Mission:** Hidden airborne irritants trigger asthma and respiratory issues. Test whether your DIY air purifier actually reduces particulate matter (PM2.5).


In [ ]:
df = pd.read_csv("data/air_quality.csv")
# Swap the path above for your real Day 2 export when it's ready — same column names.
df.head()


In [ ]:
df.describe()


**TODO:** Plot `pm25_ugm3` over `minute`, split by `condition` (before vs. after the purifier). One way:
```python
for cond, group in df.groupby("condition"):
    plt.plot(group["minute"], group["pm25_ugm3"], label=cond)
plt.legend(); plt.xlabel("Minute"); plt.ylabel("PM2.5 (µg/m³)")
plt.title("PM2.5 Before vs After Purifier")
```


In [ ]:
# TODO: your plot here



**TODO — flag the improvement:** compute the average PM2.5 for each condition and the percent reduction:
```python
avgs = df.groupby("condition")["pm25_ugm3"].mean()
pct_reduction = (avgs["before_purifier"] - avgs["after_purifier"]) / avgs["before_purifier"] * 100
```


In [ ]:
# TODO: your averages + percent reduction here



**Data story — answer these for tomorrow's pitch:**
- Did your purifier actually reduce PM2.5, and by how much (%)?
- Was the reduction steady over time, or did it vary?
- Who in your community would benefit most from a device like this?

*(Double-click this cell and type your answers right here.)*


### 🫀 Squad 3: Cardiovascular Health

**Mission:** Catch heart strain before it becomes an emergency. This sample data is shaped exactly like the [Pulse Sensor Amped](https://pulsesensor.com/products/pulse-sensor-amped) output — one row per detected heartbeat, four conditions: `resting_normal`, `exercise`, `hypertension_high_hr`, `low_tension_low_hr`.


In [ ]:
df = pd.read_csv("data/cardio.csv")
# Swap the path above for your real Day 2 export when it's ready — same column names.
df.head()


In [ ]:
df.groupby("condition")["bpm"].describe()


**TODO:** Plot `bpm` over `beat_time_s`, one line per `condition`:
```python
for cond, group in df.groupby("condition"):
    plt.plot(group["beat_time_s"], group["bpm"], label=cond)
plt.legend(); plt.xlabel("Time (s)"); plt.ylabel("BPM")
plt.title("Heart Rate by Condition")
```


In [ ]:
# TODO: your plot here



**TODO — signal quality check:** every reading also has a `noise_level` and `signal_amp` (the sensor's own confidence in that beat). Flag beats that might be unreliable — e.g. `noise_level` above 80:
```python
df[df["noise_level"] > 80]
```
Real wearables have to deal with this exact tradeoff — a heart-rate spike could be a real event, or just a noisy reading from the sensor moving.


In [ ]:
# TODO: your noisy-beat check here



**Data story — answer these for tomorrow's pitch:**
- How much higher is average BPM in `exercise` or `hypertension_high_hr` compared to `resting_normal`?
- What fraction of readings in each condition look "noisy," and does that change your confidence in the data?
- What would a concerning real-time pattern look like on a wearable like this?

*(Double-click this cell and type your answers right here.)*


### 🩹 Squad 4: Wound Care

**Mission:** Post-surgical infections delay healing. This sample data is shaped like readings from a [DS18B20 temperature sensor](https://www.analog.com/en/products/ds18b20.html) tracking a wound over 250 days, for two patients: one `healing` and one `non_healing`. `delta_c` is the temperature gap between the wound site and nearby healthy skin (the "thermal gradient").


In [ ]:
df = pd.read_csv("data/wound_care.csv")
# Swap the path above for your real Day 2 export when it's ready — same column names.
df.head()


In [ ]:
df.groupby("condition")[["delta_c", "healing_score"]].describe()


**TODO:** Plot `delta_c` over `day`, one line per `condition`:
```python
for cond, group in df.groupby("condition"):
    plt.plot(group["day"], group["delta_c"], label=cond)
plt.legend(); plt.xlabel("Day"); plt.ylabel("Thermal gradient (°C)")
plt.title("Wound Temperature Gradient Over Time")
```


In [ ]:
# TODO: your plot here



**TODO — flag a concern:** a thermal gradient that *stays* elevated (say, above 1.5°C) past the first couple of weeks can be an early infection signal. Which days/condition cross that line?
```python
df[df["delta_c"] > 1.5]
```


In [ ]:
# TODO: your flagged rows here



**Data story — answer these for tomorrow's pitch:**
- How differently does `delta_c` behave for `healing` vs `non_healing` over time?
- Around what day does the difference between the two groups become obvious?
- How might a nurse or patient be alerted to this pattern in real time?

*(Double-click this cell and type your answers right here.)*


---
## Part 5 — From Data to Pitch

Before 11:45, make sure your squad can answer, out loud, in under 2 minutes:

1. **What did you find?** (one number or one plot)
2. **Why does it matter?** (who is affected, what's the risk)
3. **What would you do next?** (this is where tomorrow's pitch idea comes from)

This is exactly the shape your Day 5 pitch should take — you're just rehearsing it today with real data behind it.
